In [1]:

import requests
import pandas as pd


#### 국토교통부 국토지리정보원_시계열행정구역
- https://www.data.go.kr/data/15122710/fileData.do?utm_source=chatgpt.com
- csv파일 다운로드
- 경로 : ``` /dataset/etc/국토교통부 국토지리정보원_시계열행정구역_20201201.csv ```


In [2]:
code_info_df = pd.read_csv('../dataset/etc/국토교통부 국토지리정보원_시계열행정구역_20201201.csv')
code_info_df = code_info_df.iloc[:,[1,2,4,5]]
code_info_df.columns = ['시도 코드', '시도 이름', '시군구 코드', '시군구 이름']
code_info_df

,시도 코드,시도 이름,시군구 코드,시군구 이름
0,11,서울특별시,11110,Jongno-gu
1,11,서울특별시,11140,Jung-gu
2,11,서울특별시,11170,Yongsan-gu
3,11,서울특별시,11200,Seongdong-gu
4,11,서울특별시,11215,Gwangjin-gu
...,...,...,...,...
224,44,충청남도,44270,Dangjin-si
225,44,충청남도,44760,Buyeo-gun
226,45,전라북도,45130,Gunsan-si
227,48,경상남도,48120,Changwon-si


한국관광공사_관광빅데이터 정보서비스_ GW

- https://www.data.go.kr/data/15101972/openapi.do

In [ ]:
decoding_key = 'YOURKEY'

In [4]:
url = 'http://apis.data.go.kr/B551011/DataLabService/metcoRegnVisitrDDList'

In [5]:

params = {
    'serviceKey' : decoding_key,
    '_type' : 'json',
    'numOfRows' : 10,
    'pageNo' : 1,
    'MobileOS' : 'ETC',
    'MobileApp' : 'AppTest',
    'startYmd' : 20230101,
    'endYmd' : 20231231

}

response = requests.get(url, params=params)
response.json()


{'response': {'header': {'resultCode': '0000', 'resultMsg': 'OK'},
  'body': {'items': {'item': [{'areaCode': '11',
      'areaNm': '서울특별시',
      'daywkDivCd': '1',
      'daywkDivNm': '월요일',
      'touDivCd': '1',
      'touDivNm': '현지인(a)',
      'touNum': '4865738.5',
      'baseYmd': '20230102'},
     {'areaCode': '11',
      'areaNm': '서울특별시',
      'daywkDivCd': '1',
      'daywkDivNm': '월요일',
      'touDivCd': '1',
      'touDivNm': '현지인(a)',
      'touNum': '4916439.5',
      'baseYmd': '20230109'},
     {'areaCode': '11',
      'areaNm': '서울특별시',
      'daywkDivCd': '1',
      'daywkDivNm': '월요일',
      'touDivCd': '1',
      'touDivNm': '현지인(a)',
      'touNum': '4904816.0',
      'baseYmd': '20230116'},
     {'areaCode': '11',
      'areaNm': '서울특별시',
      'daywkDivCd': '1',
      'daywkDivNm': '월요일',
      'touDivCd': '1',
      'touDivNm': '현지인(a)',
      'touNum': '3531069.5',
      'baseYmd': '20230123'},
     {'areaCode': '11',
      'areaNm': '서울특별시',
      'daywkDiv

In [6]:
import requests
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta

# ⛳ 디코딩된 서비스키 입력
service_key = decoding_key

# 📅 2023년 전체 날짜 생성
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)
date_list = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

# 📦 결과 저장 리스트
all_data = []

# 📡 날짜별 API 호출
for date in tqdm(date_list):
    ymd = date.strftime('%Y%m%d')
    url = "http://apis.data.go.kr/B551011/DataLabService/locgoRegnVisitrDDList"
    params = {
        'serviceKey': service_key,
        'pageNo': 1,
        'numOfRows': 1000,
        'MobileOS': 'ETC',
        'MobileApp': 'AppTest',
        '_type': 'json',
        'startYmd': ymd,
        'endYmd': ymd
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code != 200:
            print(f"{ymd} 요청 실패: {response.status_code}")
            continue

        body = response.json().get('response', {}).get('body', {})
        items_raw = body.get('items', None)

        if not items_raw:
            continue

        # 구조 분기 처리
        if isinstance(items_raw, dict):
            items = items_raw.get('item', [])
            if isinstance(items, dict):  # 단일 item
                items = [items]
        elif isinstance(items_raw, list):
            items = items_raw
        else:
            continue  # 예외 상황

        # ✅ 내국인만 수집 (현지인 or 외지인)
        for item in items:
            if item.get('touDivNm') in ['현지인(a)', '외지인(b)']:
                all_data.append({
                    'date': item.get('baseYmd'),
                    'region': item.get('signguNm'),
                    'region_code': str(item.get('signguCode')).zfill(5),  # ✅ 추가
                    'touDivNm': item.get('touDivNm'),
                    'visitors': float(item.get('touNum', 0))
                })

    except Exception as e:
        print(f"{ymd} 파싱 에러: {e}")
        continue

# 📊 DataFrame 변환
df = pd.DataFrame(all_data)

# 🧮 시군구 단위 누적 방문자 수 집계
result = df.groupby(['region', 'region_code'])['visitors'].sum().reset_index()
result = result.sort_values(by='visitors', ascending=False)


 84%|████████▍ | 306/365 [04:10<03:29,  3.55s/it]

20231102 파싱 에러: HTTPConnectionPool(host='apis.data.go.kr', port=80): Max retries exceeded with url: /B551011/DataLabService/locgoRegnVisitrDDList?serviceKey=BRq%2Fn75moMCHov%2BGcjMbuXnmY7%2Frd9NKKVlRsM08K3TBMrabTqe2bt5rTYR%2F4waqR5%2BWi%2F%2BtBPPUltQTeAeBvA%3D%3D&pageNo=1&numOfRows=1000&MobileOS=ETC&MobileApp=AppTest&_type=json&startYmd=20231102&endYmd=20231102 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x11d235e80>, 'Connection to apis.data.go.kr timed out. (connect timeout=10)'))


100%|██████████| 365/365 [04:55<00:00,  1.23it/s]


In [24]:
df.to_csv('./ref_dataset/방문자수_api.csv')

In [28]:
# 병합 전: 문자열로 타입 통일
result['region_code'] = result['region_code'].astype(str)
code_info_df['시군구 코드'] = code_info_df['시군구 코드'].astype(str)

# 병합 준비
code_info_df_subset = code_info_df[['시도 이름', '시군구 코드']].rename(columns={
    '시도 이름': 'sido_name',
    '시군구 코드': 'region_code'
})

# 병합 (left join)
merged_df = result.merge(code_info_df_subset, on='region_code', how='left')

# 병합 성공 시: '시도 이름 + 지역명', 병합 실패 시: 기존 지역명 유지
merged_df['region'] = merged_df.apply(
    lambda row: row['sido_name'] + ' ' + row['region'] if pd.notna(row['sido_name']) else row['region'],
    axis=1
)

# 불필요한 열 제거
merged_df.drop(columns=['sido_name'], inplace=True)




In [31]:
# Min-Max Scaling 적용
min_val = merged_df['visitors'].min()
max_val = merged_df['visitors'].max()

merged_df['visitors_scaled'] = (merged_df['visitors'] - min_val) / (max_val - min_val)


In [32]:
merged_df
merged_df.to_csv('./ref_dataset/04_방문자수.csv')